In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/players_clean.csv")
print(df.shape)

(18404, 16)


In [2]:
y = np.log1p(df["market_value_in_eur"])
X = df.drop(columns=["player_id", "name", "market_value_in_eur"])
X.columns.tolist()

['country_of_citizenship',
 'sub_position',
 'position',
 'foot',
 'height_in_cm',
 'international_caps',
 'international_goals',
 'current_club_domestic_competition_id',
 'caps_missing',
 'age',
 'contract_months_left',
 'no_league',
 'has_contract_info']

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(X_train.shape, X_test.shape)

(14723, 13) (3681, 13)


In [4]:
num_cols = ["age", "height_in_cm", "international_caps",
            "international_goals", "contract_months_left"]
cat_cols = ["country_of_citizenship", "position", "sub_position",
            "foot", "current_club_domestic_competition_id"]
bin_cols = ["caps_missing", "no_league", "has_contract_info"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=30,
                          sparse_output=False), cat_cols),
    ("bin", "passthrough", bin_cols),
])

In [5]:
results = []

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    mae = mean_absolute_error(y_test, pred)
    r2 = r2_score(y_test, pred)
    mae_eur = mean_absolute_error(np.expm1(y_test), np.expm1(pred))
    results.append({"model": name, "RMSE_log": rmse, "MAE_log": mae,
                    "R2_log": r2, "MAE_EUR": mae_eur})
    return pd.DataFrame(results).round(3)

In [6]:
from sklearn.dummy import DummyRegressor

baseline = Pipeline([("prep", preprocessor),
                     ("model", DummyRegressor(strategy="median"))])
evaluate("Baseline (median)", baseline)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015


In [7]:
from sklearn.linear_model import LinearRegression

linear = Pipeline([("prep", preprocessor),
                   ("model", LinearRegression())])
evaluate("Linear Regression", linear)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,1.094,0.850,0.530,2124602.263


In [8]:
from sklearn.linear_model import Ridge

ridge = Pipeline([("prep", preprocessor),
                  ("model", Ridge(alpha=1.0))])
evaluate("Ridge", ridge)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,1.094,0.850,0.530,2124602.263
2,Ridge,1.094,0.850,0.530,2124982.350


In [9]:
from sklearn.linear_model import Lasso

lasso = Pipeline([("prep", preprocessor),
                  ("model", Lasso(alpha=0.001, max_iter=10000))])
evaluate("Lasso", lasso)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,1.094,0.850,0.530,2124602.263
2,Ridge,1.094,0.850,0.530,2124982.350
3,Lasso,1.100,0.857,0.525,2133503.123


In [10]:
from sklearn.ensemble import RandomForestRegressor

forest = Pipeline([("prep", preprocessor),
                   ("model", RandomForestRegressor(n_estimators=200,
                                                   random_state=42, n_jobs=-1))])
evaluate("Random Forest", forest)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,1.094,0.850,0.530,2124602.263
2,Ridge,1.094,0.850,0.530,2124982.350
3,Lasso,1.100,0.857,0.525,2133503.123
4,Random Forest,1.024,0.787,0.588,2016816.207


In [11]:
from sklearn.ensemble import GradientBoostingRegressor

boost = Pipeline([("prep", preprocessor),
                  ("model", GradientBoostingRegressor(n_estimators=300,
                                                      learning_rate=0.1,
                                                      max_depth=3,
                                                      random_state=42))])
evaluate("Gradient Boosting", boost)

,model,RMSE_log,MAE_log,R2_log,MAE_EUR
0,Baseline (median),1.608,1.258,-0.016,2786903.015
1,Linear Regression,1.094,0.850,0.530,2124602.263
2,Ridge,1.094,0.850,0.530,2124982.350
3,Lasso,1.100,0.857,0.525,2133503.123
4,Random Forest,1.024,0.787,0.588,2016816.207
5,Gradient Boosting,0.980,0.753,0.623,1970093.845
